In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()

g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

In [7]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [8]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]
class_names_mapping = {class_names[i]:i for i in range(len(class_names))}

def perform_logistic_regression(X_train, X_val, y_train, y_val, penalty, alpha):

    if penalty == "l2":
        solver = "lbfgs"
    else:
        solver = "saga"

    logreg = LogisticRegression(max_iter=2000, class_weight="balanced",
                                penalty=penalty, solver=solver, C=1/alpha,
                                n_jobs=-1)
    logreg.fit(X_train, y_train)
    predicted = logreg.predict(X_val) # is list of strings

    predicted_num = [class_names_mapping[x] for x in predicted]
    y_val_num = [class_names_mapping[x] for x in y_val]
    print('called')

    metrics = {
        "alpha": alpha,
        "ARI": adjusted_rand_score(y_val_num, predicted_num),
        "NMI": normalized_mutual_info_score(y_val_num, predicted_num),
        "accuracy": accuracy_score(y_val_num, predicted_num),
        "macro_F1": f1_score(y_val, predicted, labels=class_names,
                                average="macro", zero_division=0),
    }

    return metrics, predicted


In [ ]:
preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

print('test')

optimize_metric = "ARI" # ARI, NMI, accuracy, macro_F1

use_features = features_C

alpha_values = [1, 2, 3, 0.5, 0.25]

X_train = preprocess.fit_transform(train[use_features])
X_val = preprocess.transform(val[use_features])
y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

alpha_idx_mapping = {alpha_values[i]:i for i in range(len(alpha_values))}

metrics_df = []

val_pred_structs = np.empty((2, len(alpha_values)), dtype=object)

for i,alpha in enumerate(alpha_values):

    metrics_L2, predicted_L2 = perform_logistic_regression(X_train, X_val, y_train, y_val, "l2", alpha)
    metrics_L1, predicted_L1 = perform_logistic_regression(X_train, X_val, y_train, y_val, "l1", alpha)
    metrics_L1["penalty"] = "l1"
    metrics_L2["penalty"] = "l2"
    metrics_df.append(metrics_L1)
    metrics_df.append(metrics_L2)
    val_pred_structs[0, i] = predicted_L1
    val_pred_structs[1, i] = predicted_L2

metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric, ascending=False)
print('metrics df:', metrics_df)

best_alpha = metrics_df["alpha"].values[0]
best_L1_L2_idx = 0 if metrics_df["penalty"].values[0] == "l1" else 1

val_predictions_best = val_pred_structs[best_L1_L2_idx, alpha_idx_mapping[best_alpha]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        val_predictions_best,
        y_val,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)

print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))

"""
metrics df:    alpha       ARI       NMI  accuracy  macro_F1 penalty
0   1.00  0.436287  0.411491  0.712679  0.552837      l1
5   3.00  0.414874  0.399965  0.595814  0.479929      l2
3   2.00  0.414297  0.399456  0.595574  0.479829      l2
1   1.00  0.414051  0.399043  0.595493  0.479722      l2
7   0.50  0.413558  0.398512  0.595293  0.479583      l2
9   0.25  0.413549  0.398535  0.595333  0.479619      l2
4   3.00  0.361555  0.342554  0.616625  0.496697      l1
6   0.50  0.348076  0.338040  0.593572  0.477917      l1
8   0.25  0.275668  0.264059  0.558913  0.435857      l1
2   2.00  0.206346  0.250456  0.343993  0.314160      l1

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                7815                1144               564   
Actual Moderate               823                5307               782   
Actual Severe                   0                1717              4680   
Actual Critical                 0                 379              1748   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   5  
Actual Severe                    17  
Actual Critical                   5

Validation classification report:
              precision    recall  f1-score   support

      Normal       0.82      0.90      0.86      8638
    Moderate       0.77      0.62      0.69      8547
      Severe       0.73      0.60      0.66      7774
    Critical       0.00      0.19      0.00        27

    accuracy                           0.71     24986
   macro avg       0.58      0.58      0.55     24986
weighted avg       0.77      0.71      0.74     24986

"""


test


/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


called


/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  w

called
called


/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  w

... (3 outputs left out)

/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  w

called
called


/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  w

called
metrics df:    alpha       ARI       NMI  accuracy  macro_F1 penalty
0   1.00  0.436287  0.411491  0.712679  0.552837      l1
5   3.00  0.414874  0.399965  0.595814  0.479929      l2
3   2.00  0.414297  0.399456  0.595574  0.479829      l2
1   1.00  0.414051  0.399043  0.595493  0.479722      l2
7   0.50  0.413558  0.398512  0.595293  0.479583      l2
9   0.25  0.413549  0.398535  0.595333  0.479619      l2
4   3.00  0.361555  0.342554  0.616625  0.496697      l1
6   0.50  0.348076  0.338040  0.593572  0.477917      l1
8   0.25  0.275668  0.264059  0.558913  0.435857      l1
2   2.00  0.206346  0.250456  0.343993  0.314160      l1

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                7815                1144               564   
Actual Moderate               823                5307               782   
Actual Severe                   0                1717              4680   
Actual Critical         